# pyAuger C++ Workflow Tutorial

This tutorial shows how to run pyAuger with the standalone C++ executables for the most expensive parts of the workflow: pair generation and matrix-element calculation.

The Python code still handles VASP parsing, carrier concentrations, NSCF input preparation, and Auger-rate analysis. The C++ executables are used where they can speed up large calculations while keeping the same pyAuger input/output files. It could speed up the calculations by ~700x.

The main route is command-line based. Python helper functions are shown only as optional alternatives.

---
## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Prerequisites](#1.-Prerequisites) | C++ source files, compiler, and build step |
| 2 | [Configuration](#2.-Configuration) | Set paths and calculation parameters |
| 3 | [Step 0 - Manual band assignment](#3.-Step-0---Manual-Band-Assignment) | Optional manual band indices |
| 4 | [Step 1 - Parse VASP data](#4.-Step-1---Parse-VASP-Data) | Read SCF VASP outputs |
| 5 | [Step 2 - Import parsed data](#5.-Step-2---Import-Parsed-Data) | Load parsed arrays |
| 6 | [Step 3 - Carrier concentrations](#6.-Step-3---Carrier-Concentrations) | Compute carrier populations |
| 7 | [Step 4 - Energy windows](#7.-Step-4---Energy-Window-Selection-(Optional)) | Select CB/VB windows |
| 8 | [Step 5 - C++ pair generation](#8.-Step-5---C++-Pair-Generation) | Choose nearest or exact k-point workflow |
| 9 | [Step 6 - C++ matrix elements](#9.-Step-6---C++-Matrix-Elements) | Prepare and run matrix elements |
| 10 | [Step 7 - Auger rates](#10.-Step-7---Auger-Rates) | Read outputs and calculate rates |

---
## 1. Prerequisites

The Python package includes the `auger/cpp` folder with:

- `pair_generation_calc.cpp`
- `matrix_element_calc.cpp`
- `prepare_pair_generation_input.py`
- `prepare_cpp_input.py`
- `Makefile`

A normal Python install does not compile the executables automatically. Build them once before using this workflow.

Check the installed C++ helper folder:

```bash
python -c "import pathlib, auger; print(pathlib.Path(auger.__file__).parent / 'cpp')"
```

Check that `g++` and `make` are available:

```bash
g++ --version
make --version
```

Common install options:

```bash
# Conda
conda install -c conda-forge compilers make

# Ubuntu / Debian
sudo apt-get install build-essential

# macOS
xcode-select --install
```

Build from the source checkout or the installed `auger/cpp` folder:

```bash
cd auger/cpp
make
ls -lh pair_generation_calc matrix_element_calc (optional)
```

---
## 2. Configuration

Edit the cell below to match your system. Do not edit the derived C++ output names later in the notebook; they are generated from these inputs.

In [ ]:
from pathlib import Path
import json

from auger import AugerCalculator, utilities

# Paths
VASP_FOLDER = "../../test-files/InAs/exact-kpoint-scf-3"
RESULTS_DIR = "./cpp_results/exact_kpoint"

# C++ executable folder. Use the installed auger/cpp path when running outside the source checkout.
CPP_DIR = "../../auger/cpp"
PAIR_EXE = "../../auger/cpp/pair_generation_calc.exe" # use pair_generation_calc on Linux/macOS
MATRIX_EXE = "../../auger/cpp/matrix_element_calc.exe" # use matrix_element_calc on Linux/macOS

# Workflow
AUGER_TYPE = "eeh"              # "eeh" or "ehh"
APPROACH = "exact_kpoint"     # "nearest_kpoint" or "exact_kpoint"

# Physical parameters
TEMPERATURE = 300               # K
DOPING = 0                      # cm^-3
EXCESS_CARRIER = 1e17           # cm^-3

# Material/calculation parameters
DIELECTRIC = 12.3               # scalar or 3x3 Cartesian tensor
FIRST_CB_INDEX = 9              # 0-based
LAST_VB_INDEX = 8               # 0-based
FORCE_GAP = 0.4                 # eV; set to None to use the parsed gap
NUM_PAIRS_TO_KEEP = "all"       # "all" or an integer
NUM_MATRIX_ELEMENTS = "all"     # "all" or an integer
NKPOINTS_PER_NSCF = 20000       # exact_kpoint only; "all" or an integer
IS_EXPANDED_FROM_IRREDUCIBLE = False

# Matrix-element execution option
THREADS = 8
USE_MATRIX_CONFIG = False
USE_CPP_SRUN = True
CPP_SRUN = {
    "srun_command": "srun",
    "srun_num_nodes": 1,
    "srun_cpu_bind": "cores",
    "srun_extra_args": [],
}
MATRIX_CONFIG_TEMPLATE = "./cpp_matrix_elements_config_example.json"


Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
CPP_WORK_DIR = f"{RESULTS_DIR}/cpp_work"
Path(CPP_WORK_DIR).mkdir(parents=True, exist_ok=True)
NSCF_DIR = f"{RESULTS_DIR}/{AUGER_TYPE}_NSCF"

# Helper functions
def cli_text(value):
    return str(value).replace("\\", "/")

def sh(value):
    text = cli_text(value)
    return f'"{text}"' if any(ch.isspace() for ch in text) else text

def print_cli(parts):
    print(" ".join(sh(p) for p in parts if p is not None))

def cpp_matrix_command(base_command):
    if not USE_CPP_SRUN:
        return base_command
    prefix = [
        CPP_SRUN.get("srun_command", "srun"),
        "-n", CPP_SRUN.get("srun_num_nodes", 1),
        "-c", THREADS,
    ]
    cpu_bind = CPP_SRUN.get("srun_cpu_bind")
    if cpu_bind:
        prefix.append(f"--cpu-bind={cpu_bind}")
    prefix.extend(CPP_SRUN.get("srun_extra_args", []))
    return prefix + base_command

def numeric_suffix(path):
    try:
        return int(path.stem.rsplit("_", 1)[-1])
    except ValueError:
        return 0


### Choosing `THREADS`

`THREADS` is used by `matrix_element_calc`. Pair generation is serial, so `THREADS` does not speed up `pair_generation_calc`.

Start with 2 to 8 threads on a laptop or small workstation. On a larger workstation, start with the number of physical CPU cores you can dedicate to the job. Do not set `THREADS` higher than the CPU cores available to the process.

---
## 3. Step 0 - Manual Band Assignment

For zero-gap or narrow-gap materials, assign the first conduction band and last valence band manually before parsing.

In [ ]:
calc = AugerCalculator(T=TEMPERATURE, nd=DOPING)
calc.assign_firstCB_and_lastVB(
    firstCB_index=FIRST_CB_INDEX,
    lastVB_index=LAST_VB_INDEX,
)


---
## 4. Step 1 - Parse VASP Data

This reads the SCF VASP output and writes pyAuger parsed arrays into `RESULTS_DIR`.

In [ ]:
calc.parse_BS_data(
    folder_path=VASP_FOLDER,
    write_path=RESULTS_DIR,
    force_gap=FORCE_GAP,
)


---
## 5. Step 2 - Import Parsed Data

Import parsed data before carrier concentrations, pair generation, or matrix-element preparation.

In [ ]:
calc.import_parsed_BS_data(from_folder=RESULTS_DIR)

# Derived C++ file names. These are intentionally generated automatically.
exact_input_bin = f"{CPP_WORK_DIR}/{AUGER_TYPE}_exact_kpoints_input.bin"
exact_csv_base = f"{RESULTS_DIR}/exact_kpoints_{AUGER_TYPE}_{calc.XX}.csv"
pair_input_bin = f"{CPP_WORK_DIR}/{AUGER_TYPE}_{APPROACH}_pairs_input.bin"
pair_csv_base = f"{RESULTS_DIR}/auger_{AUGER_TYPE}_pairs_{calc.XX}.csv"
matrix_input_bin = f"{CPP_WORK_DIR}/{AUGER_TYPE}_{APPROACH}_matrix_input.bin"
matrix_jsonl = f"{RESULTS_DIR}/{AUGER_TYPE}_matrix_elements_{calc.XX}_1.jsonl"
matrix_config = f"{CPP_WORK_DIR}/{AUGER_TYPE}_{APPROACH}_matrix_config.json"

print(f"APPROACH: {APPROACH}")
print(f"C++ work folder: {CPP_WORK_DIR}")


---
## 6. Step 3 - Carrier Concentrations

In [ ]:
fn, fp = calc.calculate_carrier_concentrations(delta_n=EXCESS_CARRIER)
print(f"Electron concentration: {calc.n:.4e} cm^-3")
print(f"Hole concentration:     {calc.p:.4e} cm^-3")
print(f"Efn: {calc.Efn:.4f} eV")
print(f"Efp: {calc.Efp:.4f} eV")


---
## 7. Step 4 - Energy-Window Selection (Optional)

In [ ]:
CB_auto, VB_auto = calc.calculate_energy_cutoffs(charge_threshold=0.99)
CB_WINDOW = CB_auto
VB_WINDOW = VB_auto

print(f"CB_window = {CB_WINDOW:.4f} eV")
print(f"VB_window = {VB_WINDOW:.4f} eV")


---
## 8. Step 5 - C++ Pair Generation

Choose one approach, the same way as in `main_tutorial.ipynb`:

| Approach | Description | Extra VASP runs? |
|----------|-------------|------------------|
| `nearest_kpoint` | Maps the fourth k-point to the nearest SCF-grid point | No |
| `exact_kpoint` | Generates exact off-grid k-points, then uses NSCF calculations | Yes |

The C++ pair generator uses `Max_Heap` by default.

### 8a. Approach 1 - `nearest_kpoint`

Run these commands only when `APPROACH = "nearest_kpoint"`.

First prepare the binary input for C++ pair generation:

```bash
python -m auger.cpp.prepare_pair_generation_input \
  --task pairs \
  --approach nearest_kpoint \
  --results_folder RESULTS_DIR \
  --output CPP_WORK_DIR/eeh_nearest_kpoint_pairs_input.bin \
  --auger_type eeh \
  --CB_window 0.5 \
  --VB_window 0.5 \
  --num_to_keep all \
  --T 300 \
  --delta_n 1e17 \
  --poscar_path VASP_FOLDER/POSCAR
```

Expected output: one binary file in `CPP_WORK_DIR`, for example `eeh_nearest_kpoint_pairs_input.bin`.

Then generate the final pair CSV chunks:

```bash
./pair_generation_calc CPP_WORK_DIR/eeh_nearest_kpoint_pairs_input.bin RESULTS_DIR/auger_eeh_pairs_XX.csv
```

Expected output: `auger_eeh_pairs_XX_1.csv`, then `_2.csv`, `_3.csv`, etc. if more chunks are needed.

In [ ]:
if APPROACH == "nearest_kpoint":
    print("Prepare nearest-kpoint pair input:")
    print_cli([
        "python", "-m", "auger.cpp.prepare_pair_generation_input",
        "--task", "pairs",
        "--approach", "nearest_kpoint",
        "--results_folder", RESULTS_DIR,
        "--output", pair_input_bin,
        "--auger_type", AUGER_TYPE,
        "--CB_window", CB_WINDOW,
        "--VB_window", VB_WINDOW,
        "--num_to_keep", NUM_PAIRS_TO_KEEP,
        "--T", TEMPERATURE,
        "--delta_n", EXCESS_CARRIER,
        "--poscar_path", f"{VASP_FOLDER}/POSCAR",
    ])
    print(f"Expected output: {pair_input_bin}")

    print("\nRun C++ pair generation:")
    print_cli([PAIR_EXE, pair_input_bin, pair_csv_base])
    print(f"Expected output: {Path(pair_csv_base).with_suffix('').name}_1.csv")
else:
    print("Skipping nearest_kpoint commands because APPROACH is exact_kpoint.")


### 8b. Approach 2 - `exact_kpoint`

Run these stages only when `APPROACH = "exact_kpoint"`.

Stage 1 prepares the exact-kpoint binary input and runs C++ to write exact-kpoint CSV chunks.

```bash
python -m auger.cpp.prepare_pair_generation_input \
  --task exact_kpoints \
  --approach exact_kpoint \
  --results_folder RESULTS_DIR \
  --output CPP_WORK_DIR/eeh_exact_kpoints_input.bin \
  --auger_type eeh \
  --CB_window 0.5 \
  --VB_window 0.5 \
  --num_to_keep all \
  --T 300 \
  --delta_n 1e17 \
  --poscar_path VASP_FOLDER/POSCAR
```

Expected output: `eeh_exact_kpoints_input.bin`.

```bash
./pair_generation_calc CPP_WORK_DIR/eeh_exact_kpoints_input.bin RESULTS_DIR/exact_kpoints_eeh_XX.csv
```

Expected output: `exact_kpoints_eeh_XX_1.csv`, then `_2.csv`, `_3.csv`, etc. if needed.

In [ ]:
if APPROACH == "exact_kpoint":
    cmd = [
        "python", "-m", "auger.cpp.prepare_pair_generation_input",
        "--task", "exact_kpoints",
        "--approach", "exact_kpoint",
        "--results_folder", RESULTS_DIR,
        "--output", exact_input_bin,
        "--auger_type", AUGER_TYPE,
        "--CB_window", CB_WINDOW,
        "--VB_window", VB_WINDOW,
        "--num_to_keep", NUM_PAIRS_TO_KEEP,
        "--T", TEMPERATURE,
        "--delta_n", EXCESS_CARRIER,
        "--poscar_path", f"{VASP_FOLDER}/POSCAR",
    ]
    if IS_EXPANDED_FROM_IRREDUCIBLE:
        cmd.append("--is_expanded_from_irreducible")

    print("Prepare exact-kpoint input:")
    print_cli(cmd)
    print(f"Expected output: {exact_input_bin}")

    print("\nRun C++ exact-kpoint generation:")
    print_cli([PAIR_EXE, exact_input_bin, exact_csv_base])
    print(f"Expected output: {Path(exact_csv_base).with_suffix('').name}_1.csv")
else:
    print("Skipping exact_kpoint stage 1 because APPROACH is nearest_kpoint.")


#### Stage 2 - Create NSCF Input Directories

After the exact-kpoint CSV chunks are written, create NSCF folders. Then run VASP externally in every generated NSCF folder before Stage 3.

Expected output: folders such as `NSCF_eeh_1`, `NSCF_eeh_2`, etc. inside `NSCF_DIR`.

In [ ]:
if APPROACH == "exact_kpoint":
    exact_csvs = sorted(
        Path(RESULTS_DIR).glob(f"exact_kpoints_{AUGER_TYPE}_{calc.XX}_*.csv"),
        key=numeric_suffix,
    )
    if not exact_csvs:
        exact_csvs = sorted(Path(RESULTS_DIR).glob(f"exact_kpoints_{AUGER_TYPE}_{calc.XX}.csv"))
    if not exact_csvs:
        raise FileNotFoundError("No exact-kpoint CSV files found. Run Stage 1 first.")

    utilities.create_nscf_inputs(
        scf_folder=VASP_FOLDER,
        nscf_folder=NSCF_DIR,
        exact_kpoints_table=[p.as_posix() for p in exact_csvs],
        auger_type=AUGER_TYPE,
        num_kpoints_per_file=NKPOINTS_PER_NSCF,
        efermi=calc.E_Fermi,
    )

    print("Run VASP in these NSCF folders before Stage 3:")
    for folder in sorted(Path(NSCF_DIR).glob(f"NSCF_{AUGER_TYPE}_*"), key=numeric_suffix):
        print(f"  {folder}")
else:
    print("Skipping NSCF input creation because APPROACH is nearest_kpoint.")


#### Stage 3 - Create Final Exact-Kpoint Pair CSV Chunks

Run this only after VASP has completed in every NSCF folder.

```bash
python -m auger.cpp.prepare_pair_generation_input \
  --task pairs \
  --approach exact_kpoint \
  --results_folder RESULTS_DIR \
  --output CPP_WORK_DIR/eeh_exact_kpoint_pairs_input.bin \
  --auger_type eeh \
  --CB_window 0.5 \
  --VB_window 0.5 \
  --num_to_keep all \
  --T 300 \
  --delta_n 1e17 \
  --poscar_path VASP_FOLDER/POSCAR \
  --nscf_folders NSCF_DIR/NSCF_eeh_1 NSCF_DIR/NSCF_eeh_2 \
  --exact_kpoints_csv RESULTS_DIR/exact_kpoints_eeh_XX_1.csv
```

Expected output: `eeh_exact_kpoint_pairs_input.bin`.

```bash
./pair_generation_calc CPP_WORK_DIR/eeh_exact_kpoint_pairs_input.bin RESULTS_DIR/auger_eeh_pairs_XX.csv
```

Expected output: `auger_eeh_pairs_XX_1.csv`, then `_2.csv`, `_3.csv`, etc. if needed.

In [ ]:
if APPROACH == "exact_kpoint":
    nscf_folders = sorted(Path(NSCF_DIR).glob(f"NSCF_{AUGER_TYPE}_*"), key=numeric_suffix)
    exact_csvs = sorted(
        Path(RESULTS_DIR).glob(f"exact_kpoints_{AUGER_TYPE}_{calc.XX}_*.csv"),
        key=numeric_suffix,
    )
    if not nscf_folders:
        raise FileNotFoundError("No NSCF folders found. Run Stage 2 and complete VASP first.")
    if not exact_csvs:
        raise FileNotFoundError("No exact-kpoint CSV files found.")

    print("Prepare final exact-kpoint pair input:")
    print_cli([
        "python", "-m", "auger.cpp.prepare_pair_generation_input",
        "--task", "pairs",
        "--approach", "exact_kpoint",
        "--results_folder", RESULTS_DIR,
        "--output", pair_input_bin,
        "--auger_type", AUGER_TYPE,
        "--CB_window", CB_WINDOW,
        "--VB_window", VB_WINDOW,
        "--num_to_keep", NUM_PAIRS_TO_KEEP,
        "--T", TEMPERATURE,
        "--delta_n", EXCESS_CARRIER,
        "--poscar_path", f"{VASP_FOLDER}/POSCAR",
        "--nscf_folders", *nscf_folders,
        "--exact_kpoints_csv", *exact_csvs,
    ])
    print(f"Expected output: {pair_input_bin}")

    print("\nRun C++ final pair generation:")
    print_cli([PAIR_EXE, pair_input_bin, pair_csv_base])
    print(f"Expected output: {Path(pair_csv_base).with_suffix('').name}_1.csv")
else:
    print("Skipping exact_kpoint stage 3 because APPROACH is nearest_kpoint.")


---
## 9. Step 6 - C++ Matrix Elements

This step is common after the final pair CSV chunks exist.

Nearest-kpoint uses the SCF `WAVECAR`. Exact-kpoint uses the NSCF `WAVECAR` files.

Prepare the matrix binary input:

```bash
python -m auger.cpp.prepare_cpp_input \
  --results_folder RESULTS_DIR \
  --wavecar_files WAVECAR_FILES \
  --pairs_csv PAIR_CSV_FILES \
  --auger_type eeh \
  --dielectric 12.3 \
  --firstCB_index 9 \
  --lastVB_index 8 \
  --output CPP_WORK_DIR/eeh_nearest_kpoint_matrix_input.bin
```

Expected output: one binary file in `CPP_WORK_DIR`.

Run the C++ matrix-element executable with positional arguments:

```bash
./matrix_element_calc CPP_WORK_DIR/eeh_nearest_kpoint_matrix_input.bin RESULTS_DIR/matrix_elements_eeh_XX_1.jsonl THREADS --overwrite --progress_interval 10000
```

Or run it with a JSON config:

```bash
./matrix_element_calc --config cpp_matrix_elements_config.json
```

An editable example config is provided at `tutorials/cpp_workflow/cpp_matrix_elements_config_example.json`.


If `USE_CPP_SRUN = True`, the printed matrix command is wrapped as:

```bash
srun -n <srun_num_nodes> -c <THREADS> --cpu-bind=<srun_cpu_bind> ./matrix_element_calc ...
```

In [ ]:
pair_csvs = sorted(
    Path(RESULTS_DIR).glob(f"auger_{AUGER_TYPE}_pairs_{calc.XX}_*.csv"),
    key=numeric_suffix,
)
if not pair_csvs:
    pair_csvs = sorted(Path(RESULTS_DIR).glob(f"auger_{AUGER_TYPE}_pairs_{calc.XX}.csv"))
if not pair_csvs:
    raise FileNotFoundError("No final pair CSV files found.")

if APPROACH == "nearest_kpoint":
    wavecar_files = [f"{VASP_FOLDER}/WAVECAR"]
else:
    nscf_folders = sorted(Path(NSCF_DIR).glob(f"NSCF_{AUGER_TYPE}_*"), key=numeric_suffix)
    wavecar_files = [f"{folder.as_posix()}/WAVECAR" for folder in nscf_folders]

prepare_cmd = [
    "python", "-m", "auger.cpp.prepare_cpp_input",
    "--results_folder", RESULTS_DIR,
    "--wavecar_files", *wavecar_files,
    "--pairs_csv", *pair_csvs,
    "--auger_type", AUGER_TYPE,
    "--dielectric", json.dumps(DIELECTRIC) if isinstance(DIELECTRIC, list) else DIELECTRIC,
    "--firstCB_index", FIRST_CB_INDEX,
    "--lastVB_index", LAST_VB_INDEX,
]
prepare_cmd.extend(["--output", matrix_input_bin, "--num_matrix_elements", NUM_MATRIX_ELEMENTS])

print("Prepare matrix-element input:")
print_cli(prepare_cmd)
print(f"Expected output: {matrix_input_bin}")

config = {
    "auger_type": AUGER_TYPE,
    "input_binary": str(matrix_input_bin),
    "output_jsonl": str(matrix_jsonl),
    "num_threads": THREADS,
    "overwrite": True,
    "append": False,
    "resume": False,
    "log_file": f"{CPP_WORK_DIR}/{AUGER_TYPE}_{APPROACH}_matrix.log",
    "progress_interval": 10000,
}
Path(matrix_config).write_text(json.dumps(config, indent=2) + "\n")

print("\nRun matrix elements:")
if USE_MATRIX_CONFIG:
    print_cli(cpp_matrix_command([MATRIX_EXE, "--config", matrix_config]))
    print(f"Expected output: {Path(matrix_jsonl).name}, plus additional chunks if needed")
else:
    print_cli(cpp_matrix_command([MATRIX_EXE, matrix_input_bin, matrix_jsonl, THREADS, "--overwrite", "--progress_interval", 10000]))
    print(f"Expected output: {Path(matrix_jsonl).name}, plus additional chunks if needed")

print(f"\nGenerated config file: {matrix_config}")
print(f"Example template file: {MATRIX_CONFIG_TEMPLATE}")


---
## 10. Step 7 - Auger Rates

After the C++ matrix-element job finishes, load the pair CSV and matrix-element JSONL chunks back into pyAuger and calculate rates.

In [ ]:
calc = AugerCalculator(T=TEMPERATURE, nd=DOPING)
calc.assign_firstCB_and_lastVB(FIRST_CB_INDEX, LAST_VB_INDEX)
calc.import_parsed_BS_data(from_folder=RESULTS_DIR)
calc.calculate_carrier_concentrations(delta_n=EXCESS_CARRIER)

pair_csvs = sorted(
    Path(RESULTS_DIR).glob(f"auger_{AUGER_TYPE}_pairs_{calc.XX}_*.csv"),
    key=numeric_suffix,
)
matrix_jsonls = sorted(
    Path(RESULTS_DIR).glob(f"matrix_elements_{AUGER_TYPE}_{calc.XX}_*.jsonl"),
    key=numeric_suffix,
)

calc.read_auger_pairs([p.as_posix() for p in pair_csvs])
calc.read_matrix_elements([p.as_posix() for p in matrix_jsonls])
auger_coeff = calc.calculate_auger_rates(auger_type=AUGER_TYPE)
auger_coeff


---
## Optional: Python Helper Functions

The CLI preparation commands call Python helper functions internally. Use these helpers only when you want to prepare C++ binary inputs from a Python script or notebook.

In [ ]:
from auger.cpp.prepare_pair_generation_input import prepare as prepare_pair_generation_input
from auger.cpp.prepare_cpp_input import prepare as prepare_cpp_input

# Nearest-kpoint pair input example:
# prepare_pair_generation_input(
#     results_folder=RESULTS_DIR,
#     output=pair_input_bin,
#     task="pairs",
#     auger_type=AUGER_TYPE,
#     approach="nearest_kpoint",
#     CB_window=CB_WINDOW,
#     VB_window=VB_WINDOW,
#     num_to_keep=NUM_PAIRS_TO_KEEP,
#     T=TEMPERATURE,
#     delta_n=EXCESS_CARRIER,
#     poscar_path=f"{VASP_FOLDER}/POSCAR",
# )

# Matrix input example after pair CSV chunks exist:
# prepare_cpp_input(
#     results_folder=RESULTS_DIR,
#     wavecar_files=[cli_text(p) for p in wavecar_files],
#     pairs_csv=[p.as_posix() for p in pair_csvs],
#     auger_type=AUGER_TYPE,
#     dielectric=DIELECTRIC,
#     firstCB_index=FIRST_CB_INDEX,
#     lastVB_index=LAST_VB_INDEX,
#     output_path=matrix_input_bin,
#     num_matrix_elements=NUM_MATRIX_ELEMENTS,
# )
